# 01: Data Preparation and SQL Layer

**What this notebook establishes:** the raw dataset's shape, the cleaning ledger (every removed row accounted for), and the six showcase SQL queries with their audit purpose and results.

**Estimated runtime:** ~1 minute

All logic lives in `src/`; this notebook only calls into it and displays results (plan/00_MASTER_PLAN.md sec 5).

## 1. Setup

In [1]:
import json
import pandas as pd
from pathlib import Path

ROOT = Path('..').resolve()

## 2. The raw dataset

Online Retail II (UCI, dataset 502): two workbook sheets, combined and profiled in `src/ingest.py` (Stage 1).

In [2]:
profile = json.loads((ROOT/'reports/metrics/stage1_profile.json').read_text())
print('rows:', profile['n_rows'], ' cols:', profile['n_cols'])
pd.DataFrame(profile['columns']).T

rows: 1067371  cols: 11


,dtype,n_null,pct_null,n_unique,min,max
invoice,string,0,0.0,53628,NaN,NaN
stock_code,string,0,0.0,5305,NaN,NaN
description,string,4382,0.4105,5698,NaN,NaN
quantity,int32,0,0.0,1057,-80995.0,80995.0
invoice_date,datetime64[ns],0,0.0,47635,2009-12-01 07:45:00,2011-12-09 12:50:00
price,float32,0,0.0,2807,-53594.359375,38970.0
customer_id,Float64,243007,22.7669,5942,12346.0,18287.0
country,string,0,0.0,43,NaN,NaN
source_sheet,string,0,0.0,2,NaN,NaN
line_no,int64,0,0.0,1067371,0.0,1067370.0


## 3. Cleaning ledger

Every row removed between the raw combine and the cleaned population is counted here. `rows_in == rows_out + sum(removed)` is asserted in `checks/gate_02.py`.

In [3]:
ledger = json.loads((ROOT/'reports/metrics/cleaning_ledger.json').read_text())
pd.Series(ledger).to_frame('value')

,value
rows_in,1067371
rows_out,1013930
cancellations,19494
exact_duplicates,33947
missing_customer,243007
adjustments,4580
nonpositive_amount,6037
reconciles,True
pct_outside_business_hours_07_20,0.1873
sunday_pct_share,13.0119


**Reconciliation:** `rows_in` ({}) minus cancellations, exact duplicates and adjustments equals `rows_out`, see `ledger['reconciles']` above (`True`).

## 4. SQL layer: the six showcase queries

`sql/queries.sql` runs against `db/audit_data.db` (Stage 1, `src/sqlite_load.py`). Each query has a stated audit purpose in its header comment. Cached results:

In [4]:
for i in range(1, 7):
    p = ROOT / f'reports/metrics/sql_q{i}.csv'
    if p.exists():
        print(f'--- Q{i} ---')
        display(pd.read_csv(p).head(5))

--- Q1 ---


,country,year_month,txn_count,total_value,avg_value,min_value,max_value
0,United Kingdom,2011-11,77475,1325346.20,17.11,0.06,4781.60
1,United Kingdom,2010-11,71117,1275581.14,17.94,0.00,15818.40
2,United Kingdom,2010-12,60093,1153350.42,19.19,0.14,13541.33
3,United Kingdom,2010-10,52800,1000408.88,18.95,0.19,10468.80
4,United Kingdom,2011-10,53337,938975.00,17.60,0.06,3285.00


--- Q2 ---


,customer_id,country,invoice_count,line_count,total_value,avg_line_value,first_txn,last_txn
0,NaN,United Kingdom,2995,233252,3148203.82,13.50,2009-12-01 11:49:00,2011-12-09 10:26:00
1,18102.0,United Kingdom,145,1058,608821.65,575.45,2009-12-01 09:24:00,2011-12-09 11:50:00
2,14646.0,Netherlands,151,3849,528602.52,137.34,2009-12-02 16:52:00,2011-12-08 12:12:00
3,14156.0,EIRE,156,4048,313946.37,77.56,2009-12-01 12:30:00,2011-11-30 10:54:00
4,14911.0,EIRE,398,11245,295972.63,26.32,2009-12-01 11:41:00,2011-12-08 15:54:00


--- Q3 ---


...

,year_month,txn_count,total_value,last2day_count,pct_in_last_2_days
0,2009-12,43957,825685.76,0,0.00
1,2010-01,30638,652708.50,1393,4.55
2,2010-02,28282,553713.30,1282,4.53
3,2010-03,40364,833570.13,3684,9.13
4,2010-04,33268,681528.99,3223,9.69


--- Q5 ---


,lead_digit,observed_count,observed_pct
0,1,417487,41.5621
1,2,162781,16.2054
2,3,111995,11.1495
3,4,78872,7.8520
4,5,75292,7.4956


--- Q6 ---


,customer_id,txn_count,round_100_count,pct_round_100
0,17857.0,60,3,5.00
1,14163.0,58,2,3.45
2,18102.0,1058,31,2.93
3,12717.0,71,2,2.82
4,12801.0,50,1,2.00


**Q1** establishes the population by country/month. **Q2** shows revenue concentration across customers. **Q3** is a SQL-native duplicate-candidate screen, the same logic `src/rules.py` implements in pandas for the full pipeline. **Q4** quantifies period-end concentration for cut-off testing. **Q5/Q6** are bonus queries demonstrating Benford extraction and round-number concentration directly in SQL, useful when the only access to client data is a read-only database connection.